In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import numpy as np
import random
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

## 1、数据处理

### 1.1 导入数据

In [2]:
data = pd.read_csv('../data/translate.csv')
data.head()


,Unnamed: 0.1,Unnamed: 0,Chinese,English
0,0,1,1929年还是1989年?,1929 or 1989?
1,1,2,巴黎-随着经济危机不断加深和蔓延，整个世界一直在寻找历史上的类似事件希望有助于我们了解目前正...,PARIS – As the economic crisis deepens and wid...
2,2,3,一开始，很多人把这次危机比作1982年或1973年所发生的情况，这样得类比是令人宽心的，因为...,"At the start of the crisis, many people likene..."
3,3,4,如今人们的心情却是沉重多了，许多人开始把这次危机与1929年和1931年相比，即使一些国家政...,"Today, the mood is much grimmer, with referenc..."
4,4,5,目前的趋势是，要么是过度的克制（欧洲），要么是努力的扩展（美国）。,The tendency is either excessive restraint (Eu...


In [3]:
# Remove unnecessary column and rename columns for ease of use
data = data[['Chinese', 'English']]
data.columns = ['trg', 'src']

### 1.2 数据划分

In [4]:

# Split dataset into train and validation sets
train_data, valid_data = train_test_split(data, test_size=0.2, random_state=42)

### 1.3 分词

In [5]:
# Define tokenizer and preprocessing functions for Chinese and English
def tokenize_english(text):
    return re.findall(r"\w+|\'\w|[.!?\'\"]", text.lower())

def tokenize_chinese(text):
    return list(text)

### 1.4创建词汇表

In [6]:
# Create vocabulary for both source and target languages
class Vocabulary:
    def __init__(self):
        self.word2idx = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
        self.idx2word = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"}
        self.word_count = 4

    def add_sentence(self, sentence):
        for word in sentence:
            if word not in self.word2idx:
                self.word2idx[word] = self.word_count
                self.idx2word[self.word_count] = word
                self.word_count += 1

    def numericalize(self, sentence):
        return [self.word2idx.get(word, self.word2idx["<unk>"]) for word in sentence]

    def denumericalize(self, indices):
        return [self.idx2word[idx] for idx in indices]


In [7]:
# Initialize vocabularies
src_vocab = Vocabulary()
trg_vocab = Vocabulary()

for _, row in train_data.iterrows():
    src_vocab.add_sentence(tokenize_english(row['src']))
    trg_vocab.add_sentence(tokenize_chinese(row['trg']))


### 1.5 数据集定义

In [8]:
# Custom dataset class
class TranslationDataset(Dataset):
    def __init__(self, data, src_vocab, trg_vocab):
        self.data = data
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src = tokenize_english(self.data.iloc[idx]['src'])
        trg = tokenize_chinese(self.data.iloc[idx]['trg'])
        src_indices = [self.src_vocab.word2idx['<sos>']] + self.src_vocab.numericalize(src) + [self.src_vocab.word2idx['<eos>']]
        trg_indices = [self.trg_vocab.word2idx['<sos>']] + self.trg_vocab.numericalize(trg) + [self.trg_vocab.word2idx['<eos>']]
        return torch.tensor(src_indices), torch.tensor(trg_indices)


### 1.6 批处理

In [9]:
# Create DataLoader with padding
def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, padding_value=src_vocab.word2idx['<pad>'], batch_first=True)
    trg_batch = pad_sequence(trg_batch, padding_value=trg_vocab.word2idx['<pad>'], batch_first=True)
    return src_batch, trg_batch

In [10]:
train_dataset = TranslationDataset(train_data, src_vocab, trg_vocab)
valid_dataset = TranslationDataset(valid_data, src_vocab, trg_vocab)

In [11]:
BATCH_SIZE = 64
train_iterator = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_iterator = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2、定义模型

### 2.1 编码器+解码器

In [13]:
# Step 2: Define Seq2Seq Model
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell

### 2.2 整体网络架构

In [14]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        trg_len = trg.shape[1]
        batch_size = trg.shape[0]
        trg_vocab_size = self.decoder.embedding.num_embeddings

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)

        input = trg[:, 0]
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            top1 = output.argmax(1)
            input = trg[:, t] if random.random() < teacher_forcing_ratio else top1

        return outputs


### 2.3 超参数

In [15]:
# Step 3: Instantiate Model
INPUT_DIM = len(src_vocab.word2idx)
OUTPUT_DIM = len(trg_vocab.word2idx)
ENC_EMB_DIM = DEC_EMB_DIM = 256
HIDDEN_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = DEC_DROPOUT = 0.1

### 2.4 模型实例化

In [16]:
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HIDDEN_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HIDDEN_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

### 2.5 模型配置

In [17]:
# Step 4: Training Loop
optimizer = optim.Adam(model.parameters(), lr=0.1)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)  # Learning rate scheduler
criterion = nn.CrossEntropyLoss(ignore_index=trg_vocab.word2idx['<pad>'])  # Corrected ignore index to trg_vocab

## 3、模型训练

In [18]:
for epoch in range(10):
    model.train()
    epoch_loss = 0
    for batch in train_iterator:
        src, trg = batch
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg)
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)
        loss = criterion(output, trg)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)  # Gradient clipping
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()  # Update learning rate
    print(f'Epoch {epoch+1}, Loss: {epoch_loss/len(train_iterator)}')

Epoch 1, Loss: 42.14547432422638
Epoch 2, Loss: 12.315094547271729
Epoch 3, Loss: 9.888958539962768
Epoch 4, Loss: 9.019887895584107
Epoch 5, Loss: 8.756057958602906
Epoch 6, Loss: 8.338143138885497
Epoch 7, Loss: 8.200319480895995
Epoch 8, Loss: 7.962611169815063
Epoch 9, Loss: 7.903048772811889
Epoch 10, Loss: 7.9762462759017945


## 4、attention代码结构

In [3]:
# 编码器模型
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        return outputs, hidden, cell

# 注意力模型
class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear((hid_dim * 2), hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias=False)
        
    def forward(self, hidden, encoder_outputs):
        src_len = encoder_outputs.shape[0]
        hidden = hidden[-1].repeat(src_len, 1, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return torch.softmax(attention, dim=0)

# 解码器模型
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout, attention):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM((hid_dim * 2) + emb_dim, hid_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear((hid_dim * 2) + emb_dim + hid_dim, output_dim)
        self.softmax = nn.LogSoftmax(dim=1)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, cell, encoder_outputs):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        a = self.attention(hidden, encoder_outputs)
        a = a.unsqueeze(1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        weighted = torch.bmm(a, encoder_outputs)
        weighted = weighted.permute(1, 0, 2)
        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        embedded = embedded.squeeze(0)
        output = output.squeeze(0)
        weighted = weighted.squeeze(0)
        prediction = self.fc_out(torch.cat((output, weighted, embedded), dim=1))
        prediction = self.softmax(prediction)
        return prediction, hidden, cell

# Seq2Seq模型
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio = 0.5):
        trg_len = trg.shape[0]
        batch_size = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        encoder_outputs, hidden, cell = self.encoder(src)
        input = trg[0,:]
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[t] = output
            top1 = output.argmax(1) 
            input = trg[t] if random.random() < teacher_forcing_ratio else top1
        return outputs